<a href="https://colab.research.google.com/github/christian-ineza/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/christian-ineza/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# **1. Task type: Classification, feeding a Ranking/scoring output.**

Underneath, the model is a binary classifier (predicts is_declining_label). The deliverable an editor uses is a ranked queue sorted by that classifier's probability. Framing-ml-problems maps "which ones first?" to ranking/scoring (precision@K) and "will this one decline?" to classification — this lane is both: classifier internally, ranking externally.

In [12]:
print("Task type: binary classification, surfaced as a ranked scoring output.")


Task type: binary classification, surfaced as a ranked scoring output.


# **2. Target/proxy: is_declining_label = (trend_direction == "down")**

This is a proxy, not an observed future outcome — it's computed from trend_pct, a current-window value. Per the label trap, trend_direction and trend_pct can never be used as features once this is the label, or the model just learns to recompute its own answer. A stronger future version would use prior 90 days of features -> next 30 days actual decline.

I'm using this proxy for now because it's what's available in the starter dataset,
and it lets me build and validate the full pipeline end to end. A stronger version
for later weeks would define the label from a genuinely future window (e.g. prior
90 days of features → next 30 days actual decline), which the lane guide names as
the "stronger capstone target" versus this "beginner proxy label."

In [13]:

import pandas as pd
import os

if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/christian-ineza/flyrank-ml-internship.git
os.chdir("flyrank-ml-internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down")
print(df["is_declining_label"].value_counts())
print(f"Share declining: {df['is_declining_label'].mean():.1%}")

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 128 (delta 40), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.85 MiB | 8.61 MiB/s, done.
Resolving deltas: 100% (40/40), done.
is_declining_label
True     16262
False    13738
Name: count, dtype: int64
Share declining: 54.2%


# **3. Metric: Precision@50**

An editor with limited time reviews the top of a queue, not the whole 30,000-row
inventory — so the metric has to match that real usage, per the framing skill's
rule "which question shape maps to which metric" (ranking → precision@K).

Precision@50 asks: of the top 50 pages the system says to review first, how many
actually turned out to be true positives on the label? The starter pipeline already
computed this for me as a baseline to beat: the hand-written rule scored 0.240,
the random forest scored 0.740 (both from outputs/model_report.md).

I chose 50 specifically because it's the number already used as the reference point
in the starter pipeline's own reporting, giving me a like-for-like baseline to compare
against — not because I've yet confirmed it matches a real editor's actual weekly
capacity (that's a threshold I'd sanity-check and possibly adjust later, per the
lane guide's "thresholds are a policy choice" section).

In [14]:
baseline_p50 = 0.240
model_p50 = 0.740
print(f"Baseline Precision@50: {baseline_p50}")
print(f"Model Precision@50: {model_p50}")
print(f"Extra correct picks in top 50: {(model_p50 - baseline_p50) * 50:.0f}")


Baseline Precision@50: 0.24
Model Precision@50: 0.74
Extra correct picks in top 50: 25


# **4. Unit of analysis: one row = one page (one pseudonymized content item, `content_id`).**

Showing this as an actual dataframe below, along with two data-quality checks flagged
by the flyrank-data skill that I need to keep in mind for any future feature work:
- rate columns (`ctr`, `engagement_rate`, `scroll_rate`) are ×100 percentages, not 0-1 fractions
- `avg_position == 0` means "no position data," not literally rank zero

In [15]:
# One row = one page
print(f"Total rows (pages): {len(df)}")
print(f"Unique content_id count: {df['content_id'].nunique()}")

display(df[["content_id", "client_id", "trend_direction", "ctr", "avg_position", "impressions_90d"]].head())


print(f"\nMax ctr value (should look like a % under ~100, not a 0-1 fraction): {df['ctr'].max()}")
print(f"Rows where avg_position == 0 (means 'no data', not rank zero): {(df['avg_position'] == 0).sum()}")


Total rows (pages): 30000
Unique content_id count: 30000


,content_id,client_id,trend_direction,ctr,avg_position,impressions_90d
0,content_304f48230142,client_f369cb89fc,down,0.76,10.6,3803
1,content_a1fb4e703a9e,client_4e07408562,down,0.05,20.3,15320
2,content_9aa793d4d895,client_7f2253d7e2,down,0.09,36.5,12581
3,content_331d6c4de07b,client_19581e27de,stable,0.49,6.2,11751
4,content_d99b7a2d90ca,client_3fdba35f04,down,0.13,44.0,19140



Max ctr value (should look like a % under ~100, not a 0-1 fraction): 100.0
Rows where avg_position == 0 (means 'no data', not rank zero): 1205


# **5. Why ML earns its place here (per framing-ml-problems: "the pattern is real but too messy to write by hand — many signals, tangled, shifting over time"):**

The starter baseline is a hand-written weighted formula combining visibility, freshness
risk, position opportunity, and depth gap. It only got Precision@50 = 0.240 — about
12 of its top 50 picks right. A random forest trained on the same underlying signals
got 0.740 — about 37 of 50 right, using the same evaluation setup (client-holdout
validation, so it's not just memorizing).

That's a 3x gap using the exact same information the rule had access to. If the
signal were simple enough for an if-statement, the hand-written rule should have
come close to the model's performance — it didn't, which is the evidence (not just
an assumption) that the real decision boundary here is nonlinear and involves
interactions between signals (e.g. freshness only matters at certain traffic levels,
position opportunity depends on both position AND volume together) that a fixed
linear weighting can't represent.

In [16]:
gap = model_p50 - baseline_p50
print(f"Precision@50 gap between fixed rule and learned model: {gap:.3f}")
print(f"That's roughly {gap*50:.0f} additional correct picks in just the top 50 — evidence the pattern is too tangled for a hand-written formula alone.")


Precision@50 gap between fixed rule and learned model: 0.500
That's roughly 25 additional correct picks in just the top 50 — evidence the pattern is too tangled for a hand-written formula alone.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.